# 05 — Monitor: Continuous verification on access events

## Google Drive Setup

In [12]:
import os
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

# Check if drive is already mounted
if not os.path.exists('/content/drive'):
  drive.mount('/content/drive')
else:
  print("Drive is already mounted.")

print("Done!")

# Filepath Search
#search_term = "your_filename.ext"  # Change this
search_term = "synthetic_assets.csv"
!find /content/drive/MyDrive -maxdepth 15 -type f -iname "synthetic_assets.csv" -print


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive is already mounted.
Done!
/content/drive/MyDrive/data/synthetic_assets.csv


In [13]:
!pip -q install pandas numpy matplotlib  # safe to re-run

In [14]:
import pandas as pd
assets = pd.read_csv('/content/drive/MyDrive/data/synthetic_assets.csv')
events = pd.read_csv('/content/drive/MyDrive/data/synthetic_access_events.csv')

WEIGHTS = {'sens': {'public':5,'internal':20,'confidential':45,'restricted':65}, 'internet':25,'pii':10,'phi':15,'enc_none':15,'enc_partial':7}
def risk(row):
    s = WEIGHTS['sens'][row['sensitivity']]
    if bool(row['internet_exposed']): s += WEIGHTS['internet']
    if bool(row['has_pii']): s += WEIGHTS['pii']
    if bool(row['has_phi']): s += WEIGHTS['phi']
    if row['encryption'] == 'none': s += WEIGHTS['enc_none']
    elif row['encryption'] in ('at_rest','in_transit'): s += WEIGHTS['enc_partial']
    return min(100, int(s))

def zt_tier(r):
    if r >= 85: return 'tier0_deny_by_default'
    if r >= 65: return 'tier1_strict_conditional'
    if r >= 45: return 'tier2_strong_auth'
    return 'tier3_baseline'

assets['risk'] = assets.apply(risk, axis=1)
assets['zt_tier'] = assets['risk'].apply(zt_tier)

df = events.merge(assets[['asset_id','zt_tier']], on='asset_id', how='left')
df.head()

,ts,asset_id,principal_type,mfa,device_compliant,network_trusted,geo_anomaly,action,zt_tier
0,2026-01-13T10:38:00,a074,workload,False,True,True,False,list,tier3_baseline
1,2026-01-01T14:10:00,a010,service,True,True,False,False,write,tier3_baseline
2,2026-01-18T09:59:00,a062,service,True,True,True,False,list,tier2_strong_auth
3,2026-01-12T05:36:00,a017,workload,True,True,False,False,admin,tier3_baseline
4,2026-01-20T10:07:00,a044,human,False,False,False,False,read,tier1_strict_conditional


In [15]:
def verify(row):
    tier = row['zt_tier']
    if tier == 'tier0_deny_by_default':
        if row['geo_anomaly']: return False
        if row['action'] in ('admin','delete_attempt'): return False
        return bool(row['mfa']) and bool(row['device_compliant']) and bool(row['network_trusted'])
    if tier == 'tier1_strict_conditional':
        if row['geo_anomaly']: return False
        return bool(row['mfa']) and bool(row['device_compliant'])
    if tier == 'tier2_strong_auth':
        return bool(row['mfa'])
    return True

df['verified_ok'] = df.apply(verify, axis=1)
df['verified_ok'].mean()

np.float64(0.7383333333333333)

In [16]:
df.groupby('zt_tier')['verified_ok'].mean().sort_index()

,verified_ok
zt_tier,
tier0_deny_by_default,0.232143
tier1_strict_conditional,0.621622
tier2_strong_auth,0.747059
tier3_baseline,1.000000
